# Build Conversion Funnel
Aggregates clickstream data from the Silver Layer to create funnel metrics across the product view, cart, checkout, and purchase stages.

In [0]:
CREATE OR REPLACE TABLE ecommerce_catalog.gold.conversion_funnel AS
WITH aggregated_steps AS (
    SELECT 
        CASE event_type
            WHEN 'product_view'     THEN 1
            WHEN 'add_to_cart'      THEN 2
            WHEN 'checkout_start'   THEN 3
            WHEN 'purchase'         THEN 4   
        END AS funnel_step_order,
        event_type                         AS funnel_step,
        COUNT(DISTINCT session_id)         AS session_count
    FROM ecommerce_catalog.silver.silver_clickstream
    WHERE event_type IN ('product_view', 'add_to_cart', 'checkout_start', 'purchase')
    GROUP BY 1, 2
),
steps_with_previous AS (
    SELECT 
        funnel_step_order,
        funnel_step,
        session_count,
        LAG(session_count, 1) OVER (ORDER BY funnel_step_order ASC) AS previous_step_count,
        FIRST_VALUE(session_count) OVER (ORDER BY funnel_step_order ASC) AS top_funnel_count
    FROM aggregated_steps
)

SELECT 
    funnel_step_order,
    funnel_step,
    session_count,
    previous_step_count,
    ROUND(
        (session_count * 100.0) / NULLIF(previous_step_count, 0), 
        2) AS conversion_from_previous_step_pct,

    ROUND(
        (session_count * 100.0) / NULLIF(top_funnel_count, 0), 
        2) AS overall_conversion_rate_pct

FROM steps_with_previous
ORDER BY funnel_step_order ASC;